# Create SQLite database

The dataset PS_20174392719_1491204439457_log.csv is hosted on Kaggle as part of the "Synthetic Financial Datasets For Fraud Detection" project.

In [1]:
import sqlite3
import csv

db_path = "transactions.db"
csv_path = "PS_20174392719_1491204439457_log.csv"
table_name = "trxns"

with sqlite3.connect(db_path) as conn, open(csv_path, "r", encoding="utf-8") as f:
    reader = csv.reader(f)
    header = next(reader)

    placeholders = ", ".join(["?"] * len(header))

    column_types = {
    "step": "INTEGER",
    "type": "TEXT",
    "amount": "REAL",
    "nameOrig": "TEXT",
    "oldbalanceOrg": "REAL",
    "newbalanceOrig": "REAL",
    "nameDest": "TEXT",
    "oldbalanceDest": "REAL",
    "newbalanceDest": "REAL",
    "isFraud": "INTEGER",
    "isFlaggedFraud": "INTEGER"
}
    conn.execute(f'DROP TABLE IF EXISTS {table_name}')
    columns_with_types = ", ".join([f'"{col}" {col_type}' for col, col_type in column_types.items()])
    conn.execute(f'CREATE TABLE {table_name} ({columns_with_types})')
    conn.executemany(
        f'INSERT INTO {table_name} VALUES ({placeholders})',
        [row for row in reader if len(row) == len(header)]
    )


# Pull the data using GPU Polars

In [6]:
import polars as pl

gpu_engine = pl.GPUEngine(
    device=0, # This is the default
    raise_on_fail=True, # Fail loudly if we can't run on the GPU.
)

In [21]:
import pandas as pd

# Load data into pandas
pdf = pd.read_sql("SELECT * FROM trxns", conn)

# Convert to Polars LazyFrame
pl_trxns = pl.from_pandas(pdf).lazy()

In [22]:
type(pl_trxns)

polars.lazyframe.frame.LazyFrame

In [23]:
pl_trxns.head().collect(engine = gpu_engine)

step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
i64,str,f64,str,f64,f64,str,f64,f64,i64,i64
1,"""PAYMENT""",9839.64,"""C1231006815""",170136.0,160296.36,"""M1979787155""",0.0,0.0,0,0
1,"""PAYMENT""",1864.28,"""C1666544295""",21249.0,19384.72,"""M2044282225""",0.0,0.0,0,0
1,"""TRANSFER""",181.0,"""C1305486145""",181.0,0.0,"""C553264065""",0.0,0.0,1,0
1,"""CASH_OUT""",181.0,"""C840083671""",181.0,0.0,"""C38997010""",21182.0,0.0,1,0
1,"""PAYMENT""",11668.14,"""C2048537720""",41554.0,29885.86,"""M1230701703""",0.0,0.0,0,0


# Filter and group transactions by year, month, and type, then compute total, average, and count of amounts, returning only high-value groups

In [24]:
%%time

# Complex Query:
result = (
    pl_trxns
    # Filter for transfers and cashouts with high amounts
    .filter(
        ((pl.col("type") == "TRANSFER") | (pl.col("type") == "CASH_OUT")) &
        (pl.col("amount") > 10000)
    )
    # Create a new column: difference between old and new balance
    .with_columns(
        (pl.col("oldbalanceOrg") - pl.col("newbalanceOrig")).alias("org_balance_diff"),
        (pl.col("oldbalanceDest") - pl.col("newbalanceDest")).alias("dest_balance_diff")
    )
    # Filter where the destination balance difference is zero (potentially suspicious)
    .filter(pl.col("dest_balance_diff") == 0)
    # Group by origin account and calculate total transferred amount and number of transactions
    .group_by("nameOrig")
    .agg([
        pl.col("amount").sum().alias("total_transferred"),
        pl.len().alias("num_transactions"),
        pl.col("isFraud").max().alias("any_fraud")
    ])
    # Sort by total transferred amount, descending
    .sort("total_transferred", descending=True)
    .collect(engine = gpu_engine)
)

print(result)


shape: (5_543, 4)
┌─────────────┬───────────────────┬──────────────────┬───────────┐
│ nameOrig    ┆ total_transferred ┆ num_transactions ┆ any_fraud │
│ ---         ┆ ---               ┆ ---              ┆ ---       │
│ str         ┆ f64               ┆ u32              ┆ i64       │
╞═════════════╪═══════════════════╪══════════════════╪═══════════╡
│ C1989519123 ┆ 1e7               ┆ 1                ┆ 1         │
│ C2018514833 ┆ 1e7               ┆ 1                ┆ 1         │
│ C1530285171 ┆ 1e7               ┆ 1                ┆ 1         │
│ C1277761503 ┆ 1e7               ┆ 1                ┆ 1         │
│ C2085303535 ┆ 1e7               ┆ 1                ┆ 1         │
│ …           ┆ …                 ┆ …                ┆ …         │
│ C1057819107 ┆ 10211.81          ┆ 1                ┆ 1         │
│ C1775746074 ┆ 10119.47          ┆ 1                ┆ 1         │
│ C2068651635 ┆ 10094.11          ┆ 1                ┆ 0         │
│ C1409897757 ┆ 10067.36          ┆ 1       

# Save the result back to the database

In [25]:
# Convert to pandas DataFrame
result_pdf = result.to_pandas()

# Save back to SQLite using the same connection
result_pdf.to_sql("high_value_transfers", conn, if_exists="replace", index=False)

5543

# Check tables from the database

In [26]:
tables = conn.execute("SELECT name FROM sqlite_master WHERE type = 'table';").fetchall()
print("Tables in the database:")
for table in tables:
    print(table[0])

Tables in the database:
trxns
high_value_transfers


# Close the database connection

In [28]:
conn.close()